In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dimaoshchepkov/my-data2/data.csv
/kaggle/input/datasets/dimaoshchepkov/my-data/data.csv


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import copy
from abc import ABC, abstractmethod
import xgboost as xgb
import random

In [12]:
def seed_everything(seed=42):
    random.seed(seed)                               
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(42)

In [13]:
class UncertaintyRegressor(ABC):

    def __init__(self, seq_len=None):
        self.seq_len = seq_len

    @abstractmethod
    def fit(self, X, y, *args, **kwargs):
        pass

    @abstractmethod
    def predict(self, X):
        pass

    @abstractmethod
    def predict_variance(self, X):
        pass

    def predict_std(self, X):
        return self.predict_variance(X) ** 0.5

    @abstractmethod
    def score_nll(self, X, y):
        pass


    def predict_full(self, X):
        mean = self.predict(X)
        var = self.predict_variance(X)

        return {
            "mean": mean,
            "variance": var,
            "std": np.sqrt(var)
        }
        
    def fit_and_score(
        self,
        X_train,
        y_train,
        X_val,
        y_val,
    ):
        self.fit(X_train, y_train)

        return self.score_nll(
            X_val,
            y_val
        )

Так как нефть может быть довольно волотильной, попытаемся предсказывать еще и локарифм дисперсии

In [14]:
class LinearUncertaintyRegression(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.log_var = nn.Linear(input_dim, 1)

    def forward(self, x):
        mean = self.linear(x)
        log_var = torch.clamp(self.log_var(x), min=-2.0, max=5.0) 
        return mean, log_var

Далее будем использовать GaussianNLLLoss, который учитывает дисперсию и среднее

In [15]:
class TorchLinearUncertaintyRegressor(UncertaintyRegressor):

    def __init__(
        self,
        epochs=30,
        lr=1e-3,
        batch_size=256,
        device="cpu",
        max_grad_norm=1.0,
    ):
        super().__init__(seq_len=None)
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.device = device
        self.max_grad_norm = max_grad_norm

        self.model = None

    def _build_model(self, input_dim):
        model = LinearUncertaintyRegression(input_dim).to(self.device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=self.lr
        )

        criterion = nn.GaussianNLLLoss()

        return model, optimizer, criterion

    def fit(self, X, y):

        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)

        self.model, optimizer, criterion = self._build_model(
            X.shape[1]
        )

        loader = DataLoader(
            TensorDataset(
                torch.tensor(X),
                torch.tensor(y),
            ),
            batch_size=self.batch_size,
            shuffle=False,
        )

        for _ in range(self.epochs):

            self.model.train()

            for batch_x, batch_y in loader:

                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)

                optimizer.zero_grad()

                mean, log_var = self.model(batch_x)

                log_var = torch.clamp(
                    log_var,
                    min=-10,
                    max=10
                )

                var = torch.exp(log_var)

                loss = criterion(
                    mean,
                    batch_y,
                    var
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.max_grad_norm
                )

                optimizer.step()

        return self

    def predict(self, X):

        X = np.asarray(X, dtype=np.float32)

        self.model.eval()

        with torch.no_grad():

            X_tensor = torch.tensor(
                X,
                device=self.device
            )

            mean, _ = self.model(X_tensor)

        return mean.cpu().numpy().ravel()

    def predict_variance(self, X):

        X = np.asarray(X, dtype=np.float32)

        self.model.eval()

        with torch.no_grad():

            X_tensor = torch.tensor(
                X,
                device=self.device
            )

            _, log_var = self.model(X_tensor)

            log_var = torch.clamp(
                log_var,
                min=-10,
                max=10
            )

            var = torch.exp(log_var)

        return var.cpu().numpy().ravel()

    def score_nll(self, X, y):

        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)

        mean = self.predict(X)

        var = self.predict_variance(X)

        var = np.maximum(var, 1e-6)

        nll = (
            0.5 * np.log(var)
            + 0.5 * (y - mean) ** 2 / var
        )

        return float(np.mean(nll))


In [16]:
df = pd.read_csv('/kaggle/input/datasets/dimaoshchepkov/my-data2/data.csv', index_col='Date').sort_index()

In [17]:
core_features = [
      "body", "range", "close_pos",
     
      "dayofweek_sin", "dayofweek_cos",
      "dayofyear_sin", "dayofyear_cos",
      "month_sin", "month_cos",
      "time_trend",
  ]

In [18]:
from typing import List, Optional, Tuple

def train_test_val_split(
    df: pd.DataFrame,
    selected_features: List[str],
    target_col: str = "target",
    seq_len: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    
    n = len(df)
    train_size = int(n * 0.70)
    val_size = int(n * 0.15)

    df_train = df.iloc[:train_size]
    df_val = df.iloc[train_size: train_size + val_size]
    df_test = df.iloc[train_size + val_size:]

    if seq_len is None:
        X_train = df_train[selected_features].values.astype(np.float32)
        y_train = df_train[target_col].values.astype(np.float32)

        X_val = df_val[selected_features].values.astype(np.float32)
        y_val = df_val[target_col].values.astype(np.float32)

        X_test = df_test[selected_features].values.astype(np.float32)
        y_test = df_test[target_col].values.astype(np.float32)

        return X_train, y_train, X_val, y_val, X_test, y_test

    def make_seq(data: pd.DataFrame):
        X = data[selected_features]
        y = data[target_col]

        Xs, ys = [], []

        for i in range(len(data) - seq_len):
            Xs.append(X.iloc[i:i + seq_len].to_numpy())
            ys.append(y.iloc[i + seq_len])

        return (
            np.array(Xs, dtype=np.float32),
            np.array(ys, dtype=np.float32)
        )

    X_train, y_train = make_seq(df_train)
    X_val, y_val = make_seq(df_val)
    X_test, y_test = make_seq(df_test)

    return X_train, y_train, X_val, y_val, X_test, y_test


In [19]:
def make_lstm_dataset(df, feature_cols, target_col, seq_len):

    X_raw = df[feature_cols].values.astype(np.float32)
    y_raw = df[target_col].values.astype(np.float32)

    Xs, ys = [], []

    for i in range(len(df) - seq_len):
        Xs.append(X_raw[i:i + seq_len])
        ys.append(y_raw[i + seq_len])

    return np.array(Xs), np.array(ys)

In [20]:
def evaluate_feature_set(feature_cols, df, model, target_col='target'):

    if model.seq_len is None:
        X_train, y_train, X_val, y_val, _, _ = train_test_val_split(df, feature_cols)

    else:
        X_train, y_train = make_lstm_dataset(df.iloc[:int(len(df)*0.7)], feature_cols, target_col, model.seq_len)

        X_val, y_val = make_lstm_dataset(df.iloc[int(len(df)*0.7):int(len(df)*0.85)], feature_cols, target_col, model.seq_len)

    model.fit(X_train, y_train)
    return model.score_nll(X_val, y_val)


def greedy_feature_selection(df, core_features, candidate_features, model: UncertaintyRegressor, target_col='target'):
    selected_features = copy.deepcopy(core_features)
    available_features = [f for f in candidate_features if f not in selected_features and f != target_col]
    
    feature_ratings = {f: 0.0 for f in available_features}
    
    print(f"\nСтарт жадного отбора признаков.")
    print(f"Начальные признаки ({len(selected_features)}): {selected_features}")
    
    print("Оценка базовой модели...")
    current_best_loss = evaluate_feature_set(selected_features, df, model)
    print(f"Начальный Val Loss: {current_best_loss:.4f}\n")
    
    iteration = 1
    TOLERANCE = 0.005
    
    while available_features and iteration < 51:
        print(f"ИТЕРАЦИЯ {iteration}")
        print(f"{'='*60}")
        
        available_features.sort(key=lambda x: feature_ratings[x], reverse=True)
        
        best_new_feature = None
        best_new_loss = float('inf')

        candidate_bar = tqdm(available_features, desc=f"Тестирование фич", unit="фича")
        
        for idx, feature in enumerate(candidate_bar):
            rating = feature_ratings[feature]
            candidate_bar.set_postfix({ 
                "Тестируем": feature, 
                "Лучший": f"{best_new_loss:.4f}"
            })
            
            test_features = selected_features + [feature]
            loss = evaluate_feature_set(test_features, df, model)

            feature_ratings[feature] += (current_best_loss - loss)
            
            if  loss < best_new_loss:
                best_new_loss = loss
                best_new_feature = feature
                candidate_bar.set_postfix({
                    "Тестируем": feature, 
                    "Лучший": f"{best_new_feature} ({best_new_loss:.4f})"
                })

            if idx >= 50 and TOLERANCE < current_best_loss - best_new_loss:
                candidate_bar.set_description("Ранний стоп: хорошая фича найдена")
                break

        print()
        
        if current_best_loss - best_new_loss > TOLERANCE:
            print(f"УСПЕХ! Добавляем '{best_new_feature}'.")
            print(f"Loss: {current_best_loss:.4f} -> {best_new_loss:.4f}\n")
            
            selected_features.append(best_new_feature)
            available_features.remove(best_new_feature)
            current_best_loss = best_new_loss
            iteration += 1
        else:
            print(f"Ни одна фича не улучшила метрику значимо.")
            print(f"Лучшая попытка: '{best_new_feature}' с потерей {best_new_loss:.4f}")
            break
        
    return selected_features

In [21]:
candidate_features = [f for f in df.columns if f != 'target' and f not in core_features]

In [22]:
linear_model = TorchLinearUncertaintyRegressor(
    epochs=30,
    lr=0.01,
    batch_size=256,
    device='cpu'
)


final_features_linear = greedy_feature_selection(
    df=df, 
    core_features=core_features, 
    candidate_features=candidate_features, 
    model=linear_model
)


Старт жадного отбора признаков.
Начальные признаки (10): ['body', 'range', 'close_pos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos', 'month_sin', 'month_cos', 'time_trend']
Оценка базовой модели...
Начальный Val Loss: 7.2988

ИТЕРАЦИЯ 1


Ранний стоп: хорошая фича найдена:   5%|▌         | 50/976 [00:17<05:15,  2.94фича/s, Тестируем=body_lag_30, Лучший=2.5074]



УСПЕХ! Добавляем 'body_lag_7'.
Loss: 7.2988 -> 2.5074

ИТЕРАЦИЯ 2


Ранний стоп: хорошая фича найдена:  33%|███▎      | 319/975 [01:43<03:33,  3.07фича/s, Тестируем=range_q25_14, Лучший=range_q25_14 (1.6121)]



УСПЕХ! Добавляем 'range_q25_14'.
Loss: 2.5074 -> 1.6121

ИТЕРАЦИЯ 3


...


УСПЕХ! Добавляем 'body_pct_range_4'.
Loss: 1.6121 -> 1.0501

ИТЕРАЦИЯ 4


Ранний стоп: хорошая фича найдена:  80%|███████▉  | 778/973 [04:13<01:03,  3.07фича/s, Тестируем=range_lag_19, Лучший=range_lag_19 (0.8457)]



УСПЕХ! Добавляем 'range_lag_19'.
Loss: 1.0501 -> 0.8457

ИТЕРАЦИЯ 5


Тестирование фич: 100%|██████████| 972/972 [05:13<00:00,  3.10фича/s, Тестируем=ad, Лучший=2.1499]                                         


Ни одна фича не улучшила метрику значимо.
Лучшая попытка: 'close_pos_diff_1' с потерей 2.1499


In [23]:
final_features_linear

['body',
 'range',
 'close_pos',
 'dayofweek_sin',
 'dayofweek_cos',
 'dayofyear_sin',
 'dayofyear_cos',
 'month_sin',
 'month_cos',
 'time_trend',
 'body_lag_7',
 'range_q25_14',
 'body_pct_range_4',
 'range_lag_19']

In [24]:
class XGBUncertaintyRegressor(UncertaintyRegressor):
    def __init__(self, **xgb_params):
        super().__init__(seq_len=None)
        self.xgb_params = xgb_params

    def fit(
        self,
        X,
        y,
        X_val=None,
        y_val=None
    ):
        self.mean_model_ = xgb.XGBRegressor(**self.xgb_params)

        fit_kwargs = {"verbose": False}

        if X_val is not None and y_val is not None:
            fit_kwargs["eval_set"] = [(X_val, y_val)]

        self.mean_model_.fit(
            X,
            y,
            **fit_kwargs
        )

        y_pred = self.mean_model_.predict(X)

        residuals_sq = np.maximum(
            (y - y_pred) ** 2,
            1e-6
        )

        log_var = np.log(residuals_sq)

        self.var_model_ = xgb.XGBRegressor(**self.xgb_params)

        fit_kwargs = {"verbose": False}

        if X_val is not None and y_val is not None:
            y_val_pred = self.mean_model_.predict(X_val)

            residuals_sq_val = np.maximum(
                (y_val - y_val_pred) ** 2,
                1e-6
            )

            log_var_val = np.log(residuals_sq_val)

            fit_kwargs["eval_set"] = [
                (X_val, log_var_val)
            ]

        self.var_model_.fit(
            X,
            log_var,
            **fit_kwargs
        )

        return self

    def predict(self, X):
        return self.mean_model_.predict(X)

    def predict_variance(self, X):
        log_var = self.var_model_.predict(X)
        log_var = np.clip(log_var, -10, 10)

        return np.exp(log_var)

    def predict_std(self, X):
        return np.sqrt(self.predict_variance(X))



    def score_nll(self, X, y):
        mean = self.predict(X)

        log_var = self.var_model_.predict(X)
        log_var = np.clip(log_var, -10, 10)

        var = np.exp(log_var)

        nll = (
            0.5 * log_var
            + 0.5 * ((y - mean) ** 2) / var
        )

        return float(np.mean(nll))

In [25]:
xgb_model = XGBUncertaintyRegressor(n_estimators=50)
final_features_xgb = greedy_feature_selection(df, core_features, candidate_features, model=xgb_model)


Старт жадного отбора признаков.
Начальные признаки (10): ['body', 'range', 'close_pos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos', 'month_sin', 'month_cos', 'time_trend']
Оценка базовой модели...
Начальный Val Loss: 185.1766

ИТЕРАЦИЯ 1


Ранний стоп: хорошая фича найдена:   5%|▌         | 50/976 [00:06<01:51,  8.32фича/s, Тестируем=body_lag_30, Лучший=98.7395]



УСПЕХ! Добавляем 'body_diff_1'.
Loss: 185.1766 -> 98.7395

ИТЕРАЦИЯ 2


Ранний стоп: хорошая фича найдена:  24%|██▍       | 233/975 [00:29<01:34,  7.85фича/s, Тестируем=body_range_21, Лучший=body_range_21 (83.8984)]



УСПЕХ! Добавляем 'body_range_21'.
Loss: 98.7395 -> 83.8984

ИТЕРАЦИЯ 3


...


УСПЕХ! Добавляем 'month_start'.
Loss: 66.9038 -> 65.2237

ИТЕРАЦИЯ 6


Ранний стоп: хорошая фича найдена:   5%|▌         | 50/971 [00:07<02:18,  6.65фича/s, Тестируем=range_pacf_lag_30, Лучший=47.2948]



УСПЕХ! Добавляем 'quarter_end'.
Loss: 65.2237 -> 47.2948

ИТЕРАЦИЯ 7


Тестирование фич: 100%|██████████| 970/970 [02:32<00:00,  6.36фича/s, Тестируем=body_lag_18, Лучший=47.2948]                  


Ни одна фича не улучшила метрику значимо.
Лучшая попытка: 'is_us_holiday' с потерей 47.2948


In [26]:
final_features_xgb

['body',
 'range',
 'close_pos',
 'dayofweek_sin',
 'dayofweek_cos',
 'dayofyear_sin',
 'dayofyear_cos',
 'month_sin',
 'month_cos',
 'time_trend',
 'body_diff_1',
 'body_range_21',
 'range_median_7',
 'body_tema_20',
 'month_start',
 'quarter_end']

In [27]:
class LSTMUncertainty(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 32, num_layers: int = 2, 
                 dense_dim: int = 16, dropout_rate: float = 0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0 
        )
        
        self.dense = nn.Linear(hidden_dim, dense_dim)
        
        self.relu = nn.ReLU()
        
        self.final_dropout = nn.Dropout(dropout_rate)
        
        self.fc_mean = nn.Linear(dense_dim, 1)
        self.fc_log_var = nn.Linear(dense_dim, 1)

    def forward(self, x):

        lstm_out, _ = self.lstm(x)
        
        last_time_step_out = lstm_out[:, -1, :] 
        
        x_dense = self.dense(last_time_step_out)
        x_activated = self.relu(x_dense)
        
        x_dropped = self.final_dropout(x_activated)
        
        mean = self.fc_mean(x_dropped)
        log_var = self.fc_log_var(x_dropped)
        
        return mean, log_var

In [28]:
class TorchLSTMUncertaintyRegressor(UncertaintyRegressor):

    def __init__(
        self,
        seq_len,
        hidden_dim=32,
        num_layers=2,
        dense_dim=16,
        dropout_rate=0.2,
        epochs=30,
        lr=1e-3,
        batch_size=256,
        device="cpu",
        max_grad_norm=1.0,
    ):
        super().__init__(seq_len=seq_len)
        
        self.seq_len = seq_len

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dense_dim = dense_dim
        self.dropout_rate = dropout_rate

        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.device = device
        self.max_grad_norm = max_grad_norm

        self.model = None

    def _build_model(self, input_dim):

        model = LSTMUncertainty(
            input_dim=input_dim,
            hidden_dim=self.hidden_dim,
            num_layers=self.num_layers,
            dense_dim=self.dense_dim,
            dropout_rate=self.dropout_rate
        ).to(self.device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.lr,
            weight_decay=1e-4
        )

        criterion = nn.GaussianNLLLoss(full=True)

        return model, optimizer, criterion


    def fit(self, X, y):

        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)

        self.model, optimizer, criterion = self._build_model(
            X.shape[2]
        )

        loader = DataLoader(
            TensorDataset(
                torch.tensor(X),
                torch.tensor(y)
            ),
            batch_size=self.batch_size,
            shuffle=False
        )

        for _ in range(self.epochs):

            self.model.train()

            for batch_x, batch_y in loader:

                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)

                optimizer.zero_grad()

                mean, log_var = self.model(batch_x)

                log_var = torch.clamp(
                    log_var,
                    min=-10,
                    max=10
                )

                var = torch.exp(log_var)

                loss = criterion(
                    mean,
                    batch_y,
                    var
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.max_grad_norm
                )

                optimizer.step()

        return self

    def predict(self, X):
        
        self.model.eval()

        with torch.no_grad():

            X_tensor = torch.tensor(
                X,
                device=self.device
            )

            mean, _ = self.model(X_tensor)

        return mean.cpu().numpy().ravel()

    def predict_variance(self, X):

        self.model.eval()

        with torch.no_grad():

            X_tensor = torch.tensor(
                X,
                device=self.device
            )

            _, log_var = self.model(X_tensor)

            log_var = torch.clamp(
                log_var,
                min=-10,
                max=10
            )

            var = torch.exp(log_var)

        return var.cpu().numpy().ravel()

    def score_nll(self, X, y):

        mean = self.predict(X)

        var = self.predict_variance(X)
        var = np.maximum(var, 1e-6)

        y = np.asarray(y)

        nll = (
            0.5 * np.log(var)
            + 0.5 * (y - mean) ** 2 / var
        )

        return float(np.mean(nll))

In [29]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [30]:


lstm_model = TorchLSTMUncertaintyRegressor(
    seq_len=14,
    hidden_dim=32,
    num_layers=2,
    dense_dim=16,
    dropout_rate=0.2,
    epochs=40,
    lr=0.001,
    batch_size=64,
    device='cuda'
)


final_features_lstm = greedy_feature_selection(
    df=df, 
    core_features=core_features, 
    candidate_features=candidate_features, 
    model=lstm_model,
    target_col='target'
)


Старт жадного отбора признаков.
Начальные признаки (10): ['body', 'range', 'close_pos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos', 'month_sin', 'month_cos', 'time_trend']
Оценка базовой модели...
Начальный Val Loss: 0.8576

ИТЕРАЦИЯ 1


Ранний стоп: хорошая фича найдена:   5%|▌         | 50/976 [01:43<32:05,  2.08s/фича, Тестируем=body_lag_30, Лучший=0.8274] 



УСПЕХ! Добавляем 'Volume_diff_1'.
Loss: 0.8576 -> 0.8274

ИТЕРАЦИЯ 2


Ранний стоп: хорошая фича найдена:   5%|▌         | 50/975 [01:45<32:29,  2.11s/фича, Тестируем=range_log_lag_8, Лучший=0.8178] 



УСПЕХ! Добавляем 'range_log_lag_1'.
Loss: 0.8274 -> 0.8178

ИТЕРАЦИЯ 3


Ранний стоп: хорошая фича найдена:   5%|▌         | 50/974 [01:44<32:18,  2.10s/фича, Тестируем=range_pct_lag_4, Лучший=0.7858]



УСПЕХ! Добавляем 'range_log_lag_9'.
Loss: 0.8178 -> 0.7858

ИТЕРАЦИЯ 4


Тестирование фич: 100%|██████████| 973/973 [33:22<00:00,  2.06s/фича, Тестируем=body_pct_lag_19, Лучший=0.7826]                         


Ни одна фича не улучшила метрику значимо.
Лучшая попытка: 'close_pos_ema_20' с потерей 0.7826


In [31]:
final_features_lstm

['body',
 'range',
 'close_pos',
 'dayofweek_sin',
 'dayofweek_cos',
 'dayofyear_sin',
 'dayofyear_cos',
 'month_sin',
 'month_cos',
 'time_trend',
 'Volume_diff_1',
 'range_log_lag_1',
 'range_log_lag_9']

In [32]:
import optuna
import xgboost as xgb
import numpy as np
optuna.logging.set_verbosity(optuna.logging.WARNING)



def objective(trial, model_class, X_train, y_train, X_val, y_val, param_space_fn):

    params = param_space_fn(trial)

    model = model_class(**params)

    model.fit(X_train, y_train)

    return model.score_nll(X_val, y_val)

In [33]:
def xgb_param_space(trial):
    return {
        "n_estimators": trial.suggest_categorical("n_estimators", [50, 100, 300, 1000]),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
    }

def linear_param_space(trial):
    return {
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        "epochs": trial.suggest_int("epochs", 20, 80),
    }

In [34]:
X_train, y_train, X_val, y_val, X_test, y_test = train_test_val_split(df, final_features_xgb)
study_xgb = optuna.create_study(direction="minimize")

study_xgb.optimize(
    lambda trial: objective(
        trial,
        XGBUncertaintyRegressor,
        X_train, y_train,
        X_val, y_val,
        xgb_param_space
    ),
    n_trials=50,
    show_progress_bar=True,
)

print(f"Лучший NLL Loss: {study_xgb.best_value:.4f}")
print("Лучшие гиперпараметры:")
for k, v in study_xgb.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]

Лучший NLL Loss: 1.2656
Лучшие гиперпараметры:
  n_estimators: 50
  learning_rate: 0.012541675796038921
  max_depth: 2
  min_child_weight: 7
  subsample: 0.918548413001379
  colsample_bytree: 0.7801933248661533
  reg_alpha: 0.009713457666634036
  reg_lambda: 0.16818729603421023


In [35]:
xgb_model = XGBUncertaintyRegressor(**study_xgb.best_params)
final_features_xgb = greedy_feature_selection(df, core_features, candidate_features, model=xgb_model)


Старт жадного отбора признаков.
Начальные признаки (10): ['body', 'range', 'close_pos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos', 'month_sin', 'month_cos', 'time_trend']
Оценка базовой модели...
Начальный Val Loss: 1.2600

ИТЕРАЦИЯ 1


Ранний стоп: хорошая фича найдена:   5%|▌         | 50/976 [00:01<00:32, 28.66фича/s, Тестируем=body_lag_30, Лучший=1.2385]



УСПЕХ! Добавляем 'upper_wick'.
Loss: 1.2600 -> 1.2385

ИТЕРАЦИЯ 2


Ранний стоп: хорошая фича найдена:   5%|▌         | 50/975 [00:01<00:32, 28.69фича/s, Тестируем=range_log_lag_9, Лучший=1.2174]



УСПЕХ! Добавляем 'body_log_lag_2'.
Loss: 1.2385 -> 1.2174

ИТЕРАЦИЯ 3


...


УСПЕХ! Добавляем 'range_range_7'.
Loss: 0.9652 -> 0.9482

ИТЕРАЦИЯ 10


Ранний стоп: хорошая фича найдена:  22%|██▏       | 208/967 [00:09<00:33, 22.50фича/s, Тестируем=close_pos_std_5, Лучший=close_pos_std_5 (0.9413)]



УСПЕХ! Добавляем 'close_pos_std_5'.
Loss: 0.9482 -> 0.9413

ИТЕРАЦИЯ 11


Тестирование фич: 100%|██████████| 966/966 [00:41<00:00, 23.37фича/s, Тестируем=range_lag_14, Лучший=0.9393]                         


Ни одна фича не улучшила метрику значимо.
Лучшая попытка: 'body_pct' с потерей 0.9393


In [36]:
X_train, y_train, X_val, y_val, X_test, y_test = train_test_val_split(df, final_features_xgb)
study_xgb = optuna.create_study(direction="minimize")

study_xgb.optimize(
    lambda trial: objective(
        trial,
        XGBUncertaintyRegressor,
        X_train, y_train,
        X_val, y_val,
        xgb_param_space
    ),
    n_trials=50,
    show_progress_bar=True,
)

print(f"Лучший NLL Loss: {study_xgb.best_value:.4f}")
print("Лучшие гиперпараметры:")
for k, v in study_xgb.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]

Лучший NLL Loss: 0.9977
Лучшие гиперпараметры:
  n_estimators: 50
  learning_rate: 0.01678416474826474
  max_depth: 2
  min_child_weight: 5
  subsample: 0.6574194320560657
  colsample_bytree: 0.7378767380409758
  reg_alpha: 0.010353791945781461
  reg_lambda: 0.21009361783112981


In [37]:
def lstm_param_space(trial):
    return {
        "seq_len": trial.suggest_categorical("seq_len", [32]),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
        "lr": trial.suggest_categorical("lr", [0.001, 0.01, 0.1]),
        "epochs": trial.suggest_int("epochs", 30, 200),
        "device": trial.suggest_categorical("device", [device]),
    }

X, y = make_lstm_dataset(df, final_features_lstm, 'target', 32)

X_train = X[:int(len(df) * 0.70)]
y_train = y[:int(len(df) * 0.70)]

X_val = X[int(len(df) * 0.70):int(len(df) * 0.85)]
y_val = y[int(len(df) * 0.70):int(len(df) * 0.85)]

study_lstm1 = optuna.create_study(direction="minimize")

study_lstm1.optimize(
    lambda trial: objective(
        trial,
        TorchLSTMUncertaintyRegressor,
        X_train, y_train,
        X_val, y_val,
        lstm_param_space
    ),
    n_trials=20,
    show_progress_bar=True,
)

print(f"Лучший NLL Loss: {study_lstm1.best_value:.4f}")
print("Лучшие гиперпараметры:")
for k, v in study_lstm1.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/20 [00:00<?, ?it/s]

Лучший NLL Loss: 0.8058
Лучшие гиперпараметры:
  seq_len: 32
  batch_size: 32
  lr: 0.01
  epochs: 88
  device: cuda


In [38]:
def lstm_param_space(trial):
    return {
        "seq_len": trial.suggest_categorical("seq_len", [64]),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
        "lr": trial.suggest_categorical("lr", [0.001, 0.01, 0.1]),
        "epochs": trial.suggest_int("epochs", 30, 200),
        "device": trial.suggest_categorical("device", [device]),
    }

X, y = make_lstm_dataset(df, final_features_lstm, 'target', 64)

X_train = X[:int(len(df) * 0.70)]
y_train = y[:int(len(df) * 0.70)]

X_val = X[int(len(df) * 0.70):int(len(df) * 0.85)]
y_val = y[int(len(df) * 0.70):int(len(df) * 0.85)]

study_lstm2 = optuna.create_study(direction="minimize")

study_lstm2.optimize(
    lambda trial: objective(
        trial,
        TorchLSTMUncertaintyRegressor,
        X_train, y_train,
        X_val, y_val,
        lstm_param_space
    ),
    n_trials=20,
    show_progress_bar=True,
)

print(f"Лучший NLL Loss: {study_lstm2.best_value:.4f}")
print("Лучшие гиперпараметры:")
for k, v in study_lstm2.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/20 [00:00<?, ?it/s]

Лучший NLL Loss: 0.7311
Лучшие гиперпараметры:
  seq_len: 64
  batch_size: 16
  lr: 0.1
  epochs: 58
  device: cuda


In [39]:
def lstm_param_space(trial):
    return {
        "seq_len": trial.suggest_categorical("seq_len", [16]),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
        "lr": trial.suggest_categorical("lr", [0.001, 0.01, 0.1]),
        "epochs": trial.suggest_int("epochs", 30, 200),
        "device": trial.suggest_categorical("device", [device]),
    }

X, y = make_lstm_dataset(df, final_features_lstm, 'target', 16)

X_train = X[:int(len(df) * 0.70)]
y_train = y[:int(len(df) * 0.70)]

X_val = X[int(len(df) * 0.70):int(len(df) * 0.85)]
y_val = y[int(len(df) * 0.70):int(len(df) * 0.85)]

study_lstm3 = optuna.create_study(direction="minimize")

study_lstm3.optimize(
    lambda trial: objective(
        trial,
        TorchLSTMUncertaintyRegressor,
        X_train, y_train,
        X_val, y_val,
        lstm_param_space
    ),
    n_trials=20,
    show_progress_bar=True,
)

print(f"Лучший NLL Loss: {study_lstm3.best_value:.4f}")
print("Лучшие гиперпараметры:")
for k, v in study_lstm3.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/20 [00:00<?, ?it/s]

Лучший NLL Loss: 0.7984
Лучшие гиперпараметры:
  seq_len: 16
  batch_size: 16
  lr: 0.001
  epochs: 32
  device: cuda


In [40]:
X_train, y_train, X_val, y_val, X_test, y_test = train_test_val_split(df, final_features_linear)
study_linear = optuna.create_study(direction="minimize")

study_linear.optimize(
    lambda trial: objective(
        trial,
        TorchLinearUncertaintyRegressor,
        X_train, y_train,
        X_val, y_val,
        linear_param_space
    ),
    n_trials=50,
    show_progress_bar=True,
)

print(f"Лучший NLL Loss: {study_linear.best_value:.4f}")
print("Лучшие гиперпараметры:")
for k, v in study_linear.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]

Лучший NLL Loss: 2.5068
Лучшие гиперпараметры:
  lr: 0.003472824594922757
  epochs: 69


In [41]:
X_train, y_train, X_val, y_val, X_test, y_test = train_test_val_split(df, final_features_xgb)
X_train = np.concat([X_train, X_val])
y_train = np.concat([y_train, y_val])
model = XGBUncertaintyRegressor(**study_xgb.best_params)
model.fit(X_train, y_train)
predict_mean = model.predict(X_test)
predict_variance = model.predict_variance(X_test)
print(predict_mean[:10])
print(predict_variance[:10])

[-0.0051946  -0.0051946  -0.0051946   0.00959027 -0.0051946  -0.0051946
 -0.01652514 -0.0051946  -0.0051946  -0.0051946 ]
[0.5283327  0.42630228 0.50014216 0.5013199  0.47539076 0.5031002
 0.4652065  0.49700022 0.482567   0.4869551 ]


In [42]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import numpy as np
from scipy.stats import norm

def negative_log_likelihood(y_true, y_pred, variance):
    std = np.sqrt(variance)
    nll = -np.log(norm.pdf(y_true, y_pred, std))
    return np.mean(nll)

#Интересно еще посмотреть реальную прибыль, которая могла бы сделать модель
def calculate_profit(y_true, y_pred):
    position = np.sign(y_pred)

    pnl = position * y_true

    return {
        "total_return": pnl.sum(),
        "mean_return": pnl.mean(),
        "num_trades": len(pnl),
        "win_rate": (pnl > 0).sum() / len(pnl),
     }

In [43]:
print(mean_absolute_error(predict_mean, y_test))
print(mean_absolute_error(predict_mean, y_test))
print(mean_absolute_percentage_error(predict_mean, y_test))
print(r2_score(predict_mean, y_test))
print(negative_log_likelihood(y_test, predict_mean, predict_variance))
print(calculate_profit(y_test, predict_mean))

1.9422125816345215
1.9422125816345215
240.97848510742188
-47.847389221191406
4.53131456538308
{'total_return': np.float32(-6.8699684), 'mean_return': np.float32(-0.041385353), 'num_trades': 166, 'win_rate': np.float64(0.4397590361445783)}


In [44]:
# отбираем топ 20 сделок, в которых модель уверена больше всего (прибыль при этом увеличилась)
threshold = np.quantile(predict_variance, 0.2)

mask = predict_variance <= threshold

In [45]:
predict_mean_masked = predict_mean[mask]
y_test_masked = y_test[mask]
print(mean_absolute_error(predict_mean_masked, y_test_masked))
print(mean_absolute_error(predict_mean_masked, y_test_masked))
print(mean_absolute_percentage_error(predict_mean_masked, y_test_masked))
print(r2_score(predict_mean_masked, y_test_masked))
print(calculate_profit(y_test_masked, predict_mean_masked))

0.6841340661048889
0.6841340661048889
99.60896301269531
-26354.82421875
{'total_return': np.float32(9.510002), 'mean_return': np.float32(0.27970594), 'num_trades': 34, 'win_rate': np.float64(0.5588235294117647)}


In [46]:
X_train, y_train, X_val, y_val, X_test, y_test = train_test_val_split(df, final_features_linear)
X_train = np.concat([X_train, X_val])
y_train = np.concat([y_train, y_val])
model = TorchLinearUncertaintyRegressor(**study_linear.best_params)
model.fit(X_train, y_train)
predict_mean = model.predict(X_test)
predict_variance = model.predict_variance(X_test)
print(predict_mean[:10])
print(predict_variance[:10])

[0.41115686 0.05468953 0.2582026  0.40221202 0.47362962 0.42353177
 0.01241905 0.23655045 0.50439954 0.6171883 ]
[148.41316 148.41316 148.41316 148.41316 148.41316 148.41316 148.41316
 148.41316 148.41316 148.41316]


In [47]:
print(mean_absolute_error(predict_mean, y_test))
print(mean_absolute_error(predict_mean, y_test))
print(mean_absolute_percentage_error(predict_mean, y_test))
print(r2_score(predict_mean, y_test))
print(negative_log_likelihood(y_test, predict_mean, predict_variance))
print(calculate_profit(y_test, predict_mean))

1.9362921714782715
1.9362921714782715
17.549345016479492
-104.90314483642578
3.4482944904082706
{'total_return': np.float32(-5.170025), 'mean_return': np.float32(-0.031144729), 'num_trades': 166, 'win_rate': np.float64(0.5301204819277109)}


In [48]:
threshold = np.quantile(predict_variance, 0.2)

mask = predict_variance <= threshold

In [49]:
predict_mean_masked = predict_mean[mask]
y_test_masked = y_test[mask]
print(mean_absolute_error(predict_mean_masked, y_test_masked))
print(mean_absolute_error(predict_mean_masked, y_test_masked))
print(mean_absolute_percentage_error(predict_mean_masked, y_test_masked))
print(r2_score(predict_mean_masked, y_test_masked))
print(calculate_profit(y_test_masked, predict_mean_masked))

1.9362921714782715
1.9362921714782715
17.549345016479492
-104.90314483642578
{'total_return': np.float32(-5.170025), 'mean_return': np.float32(-0.031144729), 'num_trades': 166, 'win_rate': np.float64(0.5301204819277109)}


In [50]:
X, y = make_lstm_dataset(df, final_features_lstm, 'target', 64)

X_train = X[:int(len(df) * 0.85)]
y_train = y[:int(len(df) * 0.85)]

X_test = X[int(len(df) * 0.85):]
y_test = y[int(len(df) * 0.85):]

model = TorchLSTMUncertaintyRegressor(**study_lstm2.best_params)
model.fit(X_train, y_train)
predict_mean = model.predict(X_test)
predict_variance = model.predict_variance(X_test)
print(predict_mean[:10])
print(predict_variance[:10])

[-0.04426125 -0.04426125 -0.04426125 -0.04426125 -0.04426125 -0.04426125
 -0.04426125 -0.04426125 -0.04426125 -0.04426125]
[1.2953459 1.2953459 1.2953459 1.2953459 1.2953459 1.2953459 1.2953459
 1.2953459 1.2953459 1.2953459]


In [51]:
print(mean_absolute_error(predict_mean, y_test))
print(mean_absolute_error(predict_mean, y_test))
print(mean_absolute_percentage_error(predict_mean, y_test))
print(r2_score(predict_mean, y_test))
print(negative_log_likelihood(y_test, predict_mean, predict_variance))
print(calculate_profit(y_test, predict_mean))

2.6217212677001953
2.6217212677001953
59.23286819458008
-5.89885883767849e+16
6.104157476884367
{'total_return': np.float32(-6.6299706), 'mean_return': np.float32(-0.064999714), 'num_trades': 102, 'win_rate': np.float64(0.47058823529411764)}


In [52]:
threshold = np.quantile(predict_variance, 0.2)

mask = predict_variance <= threshold

In [53]:
predict_mean_masked = predict_mean[mask]
y_test_masked = y_test[mask]
print(mean_absolute_error(predict_mean_masked, y_test_masked))
print(mean_absolute_error(predict_mean_masked, y_test_masked))
print(mean_absolute_percentage_error(predict_mean_masked, y_test_masked))
print(r2_score(predict_mean_masked, y_test_masked))
print(calculate_profit(y_test_masked, predict_mean_masked))

2.6217212677001953
2.6217212677001953
59.23286819458008
-5.89885883767849e+16
{'total_return': np.float32(-6.6299706), 'mean_return': np.float32(-0.064999714), 'num_trades': 102, 'win_rate': np.float64(0.47058823529411764)}
